# Notebook 3: Universal and Fault-Tolerant Quantum Computing

This notebook is part of the FDP hands-on session on Quantum Error Correction (QEC) and Fault Tolerance. We cover:
1. **The 3-Qubit Bit-Flip Code**: Encoding, noise injection, syndrome extraction, and correction.
2. **The 3-Qubit Phase-Flip Code**: Syndrome measurement in the transverse basis.
3. **Transversal Logical Gates**: Verifying fault-tolerant gate action.

We will implement these models in both **Qiskit** and **PennyLane**.

## Section 1: The 3-Qubit Bit-Flip Code

The 3-qubit bit-flip code encodes a single logical qubit state $\alpha|0_L\rangle + \beta|1_L\rangle$ into three physical qubits:
$$|0_L\rangle = |000\rangle, \qquad |1_L\rangle = |111\rangle$$

It can detect and correct a single physical $X$ (bit-flip) error by measuring the stabilizers $S = \{Z_1 Z_2, Z_2 Z_3\}$ using two ancilla qubits.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
import numpy as np

# Setup circuit with 3 code qubits (0, 1, 2), 2 syndrome ancillas (3, 4), 
# and classical registers for the syndrome and output
qr = QuantumRegister(5)
cr_syndrome = ClassicalRegister(2, name="syndrome")
cr_output = ClassicalRegister(3, name="output")
qc = QuantumCircuit(qr, cr_syndrome, cr_output)

# 1. State Preparation (prepare arbitrary state on Q0)
theta = np.pi / 3
qc.ry(theta, 0)

# 2. Encoding
qc.cx(0, 1)
qc.cx(0, 2)
qc.barrier()

# 3. Noise Injection (introduce a bit flip error on physical qubit 1)
qc.x(1)
qc.barrier()

# 4. Syndrome Extraction
# Measure Z0Z1 parity on ancilla 3
qc.cx(0, 3)
qc.cx(1, 3)
# Measure Z1Z2 parity on ancilla 4
qc.cx(1, 4)
qc.cx(2, 4)
qc.barrier()

# Measure the syndrome ancillas
qc.measure(3, cr_syndrome[0])
qc.measure(4, cr_syndrome[1])

# Measure the physical qubits to see the error
qc.measure([0, 1, 2], cr_output)

print("Bit-Flip Code Circuit:")
print(qc.draw(output='text'))

Bit-Flip Code Circuit:
            ┌─────────┐           ░       ░                      ░       ┌─┐   »
      q0_0: ┤ Ry(π/3) ├──■────■───░───────░───■──────────────────░───────┤M├───»
            └─────────┘┌─┴─┐  │   ░ ┌───┐ ░   │                  ░       └╥┘┌─┐»
      q0_1: ───────────┤ X ├──┼───░─┤ X ├─░───┼────■────■────────░────────╫─┤M├»
                       └───┘┌─┴─┐ ░ └───┘ ░   │    │    │        ░        ║ └╥┘»
      q0_2: ────────────────┤ X ├─░───────░───┼────┼────┼────■───░────────╫──╫─»
                            └───┘ ░       ░ ┌─┴─┐┌─┴─┐  │    │   ░ ┌─┐    ║  ║ »
      q0_3: ──────────────────────░───────░─┤ X ├┤ X ├──┼────┼───░─┤M├────╫──╫─»
                                  ░       ░ └───┘└───┘┌─┴─┐┌─┴─┐ ░ └╥┘┌─┐ ║  ║ »
      q0_4: ──────────────────────░───────░───────────┤ X ├┤ X ├─░──╫─┤M├─╫──╫─»
                                  ░       ░           └───┘└───┘ ░  ║ └╥┘ ║  ║ »
syndrome: 2/════════════════════════════════════════════════════════╩══╩══╬══╬═»
     

In [2]:
# Simulate the circuit
sim = AerSimulator()
result = sim.run(qc, shots=1000).result()
counts = result.get_counts()

# In Qiskit, outcomes are printed as 'output_string syndrome_string' (little-endian)
print("Sampled counts (physical_state syndrome):")
for k, v in list(counts.items())[:5]:
    print(f"Outcome: {k} | Occurrences: {v}")

# Syndrome Decoder:
# '00' -> No error
# '01' -> Error on Q2 (Z1Z2 flipped)
# '10' -> Error on Q0 (Z0Z1 flipped)
# '11' -> Error on Q1 (both Z0Z1 and Z1Z2 flipped)
sample_outcome = list(counts.keys())[0]
syndrome = sample_outcome.split()[1] # Syndrome bits
physical = sample_outcome.split()[0] # Physical qubit state

print(f"\nDetected Syndrome: {syndrome}")
if syndrome == '00':
    print("No correction needed.")
elif syndrome == '01':
    print("Correction: Apply X on Q2.")
elif syndrome == '10':
    print("Correction: Apply X on Q0.")
elif syndrome == '11':
    print("Correction: Apply X on Q1.")

Sampled counts (physical_state syndrome):
Outcome: 101 11 | Occurrences: 249
Outcome: 010 11 | Occurrences: 751

Detected Syndrome: 11
Correction: Apply X on Q1.


### Bit-Flip Code in PennyLane

In [3]:
import pennylane as qml

dev_bf = qml.device("default.qubit", wires=5)

@qml.qnode(dev_bf)
def bit_flip_code_pl(error_qubit):
    # 1. State Preparation
    qml.RY(theta, wires=0)
    
    # 2. Encode
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[0, 2])
    
    # 3. Inject single X error
    if error_qubit in [0, 1, 2]:
        qml.PauliX(wires=error_qubit)
        
    # 4. Measure stabilizers Z0Z1 (on wire 3) and Z1Z2 (on wire 4)
    qml.CNOT(wires=[0, 3])
    qml.CNOT(wires=[1, 3])
    
    qml.CNOT(wires=[1, 4])
    qml.CNOT(wires=[2, 4])
    
    return qml.probs(wires=[3, 4])

print("PennyLane Syndrome Probabilities for error on Q1:")
probs = bit_flip_code_pl(error_qubit=1)
# Index in binary: 00 -> idx 0; 01 -> idx 1; 10 -> idx 2; 11 -> idx 3
print(f"P(00): {probs[0]:.2f} | P(01): {probs[1]:.2f} | P(10): {probs[2]:.2f} | P(11): {probs[3]:.2f}")

PennyLane Syndrome Probabilities for error on Q1:
P(00): 0.00 | P(01): 0.00 | P(10): 0.00 | P(11): 1.00


## Section 2: The 3-Qubit Phase-Flip Code

The phase-flip code encodes a logical qubit into the transverse Hadamard basis:
$$|0_L\rangle = |+++\rangle, \qquad |1_L\rangle = |---\rangle$$

It detects and corrects a single $Z$ (phase-flip) error by measuring the stabilizers $S = \{X_1 X_2, X_2 X_3\}$.

In [4]:
# Construct Phase-Flip Code in Qiskit
qr_pf = QuantumRegister(5)
cr_pf_syndrome = ClassicalRegister(2, name="syndrome")
cr_pf_output = ClassicalRegister(3, name="output")
qc_pf = QuantumCircuit(qr_pf, cr_pf_syndrome, cr_pf_output)

# 1. State preparation
qc_pf.ry(theta, 0)

# 2. Encode to bit-flip, then rotate to X-basis using H gates
qc_pf.cx(0, 1)
qc_pf.cx(0, 2)
qc_pf.h([0, 1, 2])
qc_pf.barrier()

# 3. Noise Injection: apply Z error on Q2
qc_pf.z(2)
qc_pf.barrier()

# 4. Syndrome Extraction (X0X1 and X1X2 parities)
# We map X basis to Z basis by applying H gates on code qubits 
# before entangling with syndrome ancillas, then map back.
qc_pf.h([0, 1, 2])
qc_pf.cx(0, 3)
qc_pf.cx(1, 3)
qc_pf.cx(1, 4)
qc_pf.cx(2, 4)
qc_pf.h([0, 1, 2])
qc_pf.barrier()

# Measure syndrome
qc_pf.measure(3, cr_pf_syndrome[0])
qc_pf.measure(4, cr_pf_syndrome[1])

# Measure code qubits in X-basis to verify state
qc_pf.h([0, 1, 2])
qc_pf.measure([0, 1, 2], cr_pf_output)

print("Phase-Flip Code Circuit:")
print(qc_pf.draw(output='text'))

Phase-Flip Code Circuit:
            ┌─────────┐          ┌───┐ ░       ░ ┌───┐     ┌───┐               »
      q1_0: ┤ Ry(π/3) ├──■────■──┤ H ├─░───────░─┤ H ├──■──┤ H ├───────────────»
            └─────────┘┌─┴─┐  │  ├───┤ ░       ░ ├───┤  │  └───┘     ┌───┐     »
      q1_1: ───────────┤ X ├──┼──┤ H ├─░───────░─┤ H ├──┼────■────■──┤ H ├─────»
                       └───┘┌─┴─┐├───┤ ░ ┌───┐ ░ ├───┤  │    │    │  └───┘┌───┐»
      q1_2: ────────────────┤ X ├┤ H ├─░─┤ Z ├─░─┤ H ├──┼────┼────┼────■──┤ H ├»
                            └───┘└───┘ ░ └───┘ ░ └───┘┌─┴─┐┌─┴─┐  │    │  └───┘»
      q1_3: ───────────────────────────░───────░──────┤ X ├┤ X ├──┼────┼───────»
                                       ░       ░      └───┘└───┘┌─┴─┐┌─┴─┐     »
      q1_4: ───────────────────────────░───────░────────────────┤ X ├┤ X ├─────»
                                       ░       ░                └───┘└───┘     »
syndrome: 2/═══════════════════════════════════════════════════════════════════»
   

In [5]:
# Run Phase-flip simulation
counts_pf = sim.run(qc_pf, shots=1000).result().get_counts()
sample_outcome_pf = list(counts_pf.keys())[0]
syndrome_pf = sample_outcome_pf.split()[1]

print(f"Detected Phase-Flip Syndrome: {syndrome_pf}")
if syndrome_pf == '00':
    print("No correction needed.")
elif syndrome_pf == '01':
    print("Correction: Apply Z on Q2.")
elif syndrome_pf == '10':
    print("Correction: Apply Z on Q0.")
elif syndrome_pf == '11':
    print("Correction: Apply Z on Q1.")

Detected Phase-Flip Syndrome: 10
Correction: Apply Z on Q0.


## Section 3: Transversal Logical Gates

A logical gate operation is **transversal** if it can be applied bitwise to the physical qubits in the code block without causing physical errors to propagate. 

For the 3-qubit bit-flip code, applying physical $X$ to all three physical qubits ($X^{\otimes 3}$) implements a logical $X_L$ gate transversally, because:
$$X^{\otimes 3}|000\rangle = |111\rangle, \qquad X^{\otimes 3}|111\rangle = |000\rangle$$

In [6]:
# Construct a circuit demonstrating logical X transversal action
qr_tg = QuantumRegister(3)
cr_tg = ClassicalRegister(3)
qc_tg = QuantumCircuit(qr_tg, cr_tg)

# 1. State preparation (prepare |0_L> = |000>)
# 2. Apply logical transversal X gate (apply Pauli-X to all qubits in block)
qc_tg.x([0, 1, 2])
qc_tg.measure([0, 1, 2], cr_tg)

counts_tg = sim.run(qc_tg, shots=10).result().get_counts()
print("Transversal Logical X applied to |000> yields:")
print(counts_tg)

Transversal Logical X applied to |000> yields:
{'111': 10}
